In [ ]:
import pandas as pd
import pickle


In [ ]:
df = pd.read_excel('/content/drive/MyDrive/fix_systematic_review/goal/mmat_final.xlsx')

with open("/content/drive/MyDrive/fix_systematic_review/134_texts.pkl", "rb") as f:
    file_texts = pickle.load(f)


In [ ]:
import pandas as pd
import re
from openai import OpenAI

client = OpenAI(api_key="api_key")



def call_openai(prompt, model="gpt-4o-mini"):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
         temperature=0,
    max_tokens=2048,
    top_p=0.45
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"




#Get purpose

In [ ]:
prompt_template = """
You are an expert in systematic reviews.
Your task is to extract the study's research aim or purpose, if explicitly stated in the text.

Input:
Title: {title}
Text: {text}

Respond in the exact format:
Purpose: <short description of the study's purpose or None if not stated>
"""


In [ ]:
from tqdm import tqdm
import time

# Column names
type_col = "Type_final"               # Column with MMAT type (e.g., "Qualitative", "Mixed methods")
text_col = "Matching_Filename"  # Column with the filename to get text from

# Save settings
save_interval = 10              # Save progress every N rows
output_file = "/content/drive/MyDrive/fix_systematic_review/goal/purpose.xlsx"  # Path to save progress file

df[f"purpose"] = ""

start_time = time.time()

# Loop over DataFrame rows with tqdm progress bar
for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    title = row.get('Title', None)       # Get MMAT type for this row
    number_id = row.get(text_col, None) # Get filename for this row

    prompt = prompt_template.format(text=file_texts[number_id],title= title)
    # Send prompt to the model and get the result
    result = call_openai(prompt)
    # print( result)

    # Save the result in the corresponding MMAT_Q column
    df.at[i, f"purpose"] = result
    print('-'*10)

    # Save progress every 'save_interval' rows
    if i % save_interval == 0 and i > 0:
        df.to_excel(output_file, index=False)
        print(f"💾 Progress saved at row {i} to {output_file}")
# Final save after processing all rows
df.to_excel(output_file, index=False)
print(f"✅ Processing completed. Final file saved to {output_file}")


#Get Results

In [ ]:
prompt_template = """
You are an expert in systematic reviews.
Your task is to extract the study's main results or key findings, if explicitly stated in the text.

Input:
Title: {title}
Text: {text}

Respond in the exact format:
Results: <short description of the study's main results or None if not stated>
"""

In [ ]:
from tqdm import tqdm
import time

# Column names
type_col = "Type_final"               # Column with MMAT type (e.g., "Qualitative", "Mixed methods")
text_col = "Matching_Filename"  # Column with the filename to get text from

# Save settings
save_interval = 10              # Save progress every N rows
output_file = "/content/drive/MyDrive/fix_systematic_review/goal/purpose_results.xlsx"  # Path to save progress file

df[f"main_results"] = ""

start_time = time.time()

# Loop over DataFrame rows with tqdm progress bar
for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    title = row.get('Title', None)       # Get MMAT type for this row
    number_id = row.get(text_col, None) # Get filename for this row

    prompt = prompt_template.format(text=file_texts[number_id],title= title)
    # Send prompt to the model and get the result
    result = call_openai(prompt)
    # print( result)

    # Save the result in the corresponding MMAT_Q column
    df.at[i, f"main_results"] = result
    print('-'*10)



    # Save progress every 'save_interval' rows
    if i % save_interval == 0 and i > 0:
        df.to_excel(output_file, index=False)
        print(f"💾 Progress saved at row {i} to {output_file}")
# Final save after processing all rows
df.to_excel(output_file, index=False)
print(f"✅ Processing completed. Final file saved to {output_file}")


#combine with participants and location

In [ ]:
import pandas as pd


output_file = "/content/drive/MyDrive/fix_systematic_review/location_participants/participants_gpt.xlsx"  # Path to save progress file
data = pd.read_excel(output_file)

In [ ]:
import re
pattern = r"Participants_actual:\s*([\d,]+)"

def extract_participants(lst):
    results = []
    for t in lst:
        match = re.search(pattern, t)
        if match:
            num = int(match.group(1).replace(",", ""))
            results.append(num)
        else:
            results.append(None)
    return results

participants_clean = extract_participants(data.participants.tolist())
data['Participants_extracted'] = participants_clean

In [ ]:
df['location'] = data['country_fix']
df['participants'] = data['Participants_extracted']

In [ ]:
df.to_excel('/content/drive/MyDrive/fix_systematic_review/goal/final.xlsx')

In [ ]:
df.sample(20).to_excel('/content/drive/MyDrive/fix_systematic_review/goal/final_sample_20.xlsx')

#Analysis

In [ ]:
import pandas as pd

df= pd.read_excel('/content/drive/MyDrive/fix_systematic_review/goal/final.xlsx')

In [ ]:
import pandas as pd


# Convert location strings like "['United States']" to actual Python lists
df['location'] = df['location'].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Expand rows where there are multiple countries in the "location" field
df_exploded = df.explode('location')

# 1. Count number of studies per country
country_counts = df_exploded['location'].value_counts()

# 2. Total participants per country
participants_sum = df_exploded.groupby('location')['participants'].sum().sort_values(ascending=False)

# 3. Quality distribution per country (High, Moderate, Low)
quality_by_country = df_exploded.pivot_table(
    index='location',
    columns='quality_label',
    aggfunc='size',
    fill_value=0
)

# 4. Countries with few studies but many participants
few_studies_large_samples = pd.DataFrame({
    'studies': country_counts,
    'participants': participants_sum
}).sort_values(by=['studies', 'participants'], ascending=[True, False])

print("Number of studies per country:")
print(country_counts)

print("\nTotal participants per country:")
print(participants_sum)

print("\nQuality distribution by country:")
print(quality_by_country)

print("\nCountries with few studies but large samples:")
print(few_studies_large_samples.head(10))


Number of studies per country:
location
United States           31
United Kingdom          18
Australia               12
Saudi Arabia             6
China                    5
Canada                   5
Turkey                   5
Iran                     3
Norway                   3
Ireland                  3
Taiwan                   3
Hong Kong                2
Sweden                   2
Denmark                  2
Spain                    2
Ethiopia                 2
South Africa             2
Netherlands              2
Colombia                 1
Ghana                    1
Nepal                    1
Jordan                   1
South Korea              1
United Arab Emirates     1
Bolivia                  1
Jamaica                  1
Austria                  1
Singapore                1
Kenya                    1
Nigeria                  1
Finland                  1
Malaysia                 1
Chile                    1
Tunisia                  1
Name: count, dtype: int64

Total participa

In [ ]:
import pandas as pd

# Convert location strings like "['United States']" to actual Python lists
df['location'] = df['location'].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Expand rows where there are multiple countries in the "location" field
df_exploded = df.explode('location')

# 1. Count number of studies per country
country_counts = df_exploded['location'].value_counts().rename_axis('country').reset_index(name='num_studies')

# 2. Total participants per country
participants_sum = df_exploded.groupby('location')['participants'].sum().reset_index().rename(columns={'location': 'country', 'participants': 'total_participants'})

# 3. Quality distribution per country
quality_by_country = df_exploded.pivot_table(
    index='location',
    columns='quality_label',
    aggfunc='size',
    fill_value=0
).reset_index().rename(columns={'location': 'country'})

# 4. Few studies but large samples
few_studies_large_samples = pd.DataFrame({
    'studies': df_exploded['location'].value_counts(),
    'participants': df_exploded.groupby('location')['participants'].sum()
}).sort_values(by=['studies', 'participants'], ascending=[True, False]).reset_index().rename(columns={'index': 'country'})

# ---- Export to Excel ----
output_path = "country_analysis.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    country_counts.to_excel(writer, sheet_name='Studies per country', index=False)
    participants_sum.to_excel(writer, sheet_name='Participants per country', index=False)
    quality_by_country.to_excel(writer, sheet_name='Quality distribution', index=False)
    few_studies_large_samples.to_excel(writer, sheet_name='Few studies large N', index=False)

print(f"✅ File saved as: {output_path}")


✅ File saved as: country_analysis.xlsx
